# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY, base_url=os.getenv("BASE_URL"))
tavily = TavilyClient(api_key=TAVILY_API_KEY)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game


@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most results in the vector DB
    
    Args:
        query (str): a question about game industry. 

    Returns:
        list: A list of results from the vector DB. Each element contains:
            - Platform: like Game Boy, Playstation 5, Xbox 360...)
            - Name: Name of the Game
            - YearOfRelease: Year when that game was released for that platform
            - Description: Additional details about the game
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")

    collection = chroma_client.get_collection(name="udaplay")
    
    results = collection.query(query_texts=[query], n_results=5)
        
    return results

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
@tool
def evaluate_retrieval(question: str, retrieved_docs: list):
    """
    Based on the user's question and on the list of retrieved documents, it will analyze the usability of the documents to respond to that question. 

    Args: 
        question (str): original question from user
        retrieved_docs (str): retrieved documents most similar to the user query in the Vector Database
   
    Returns:
        bool: True if the documents are useful to answer the question, false otherwise
        The resescription: description about the evaluation result
    """
    
    evaluation_prompt = f"Your task is to evaluate if the documents are enough to respond the query. The query is: {question}. The retrieved documents are: {retrieved_docs}. Return only 'Yes' or 'no'." 
    
    messages = [{"role": "user", "content": evaluation_prompt}]
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        max_tokens=5,
        temperature=0
    )
    
    return response.choices[0].message.content.strip().lower()

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str):
    """
    Semantic search: Finds most results in the vector DB
    
    Args:
        question (str): a question about game industry. 
    """

    response = tavily.search(question, max_results=3)
    
    return {
            "results": [
                {
                    "title": result['title'],
                    "url": result['url'],
                    "score": result['score']
                }
                for result in response.get("results", [])
            ]
    }

### Agent

In [7]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

instructions = """
You are UdaPlay, an analyst for the video game industry.
Answer the question using ONLY the provided context.
Always end your answer with a Sources section listing where 
the information came from — either 'Internal Database' or 
the web URL if a web search was used.
"""

tools = [retrieve_game, evaluate_retrieval, game_web_search]

agent = Agent(
    model_name="gpt-3.5-turbo",
    instructions=instructions,
    tools=tools,
    temperature=0.7)

In [8]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
    "Who wrote the story for Nier Automata?",
    "What is the most recent Star Ocean game?",
    "Who is the main character of FFVII?",
    "When was FFXI released?"
]

agent_outputs = []
i = 1

for query in queries:
    print(f"Processing Query {i}: {query}")
    session_id = f"session_{i}"
    output = agent.invoke(query=query, session_id=session_id)
    agent_outputs.append(output)
    answer = output.get_final_state()["messages"][-1].content    
    print(f"Query: {query}\nAgent Output: {answer}\n{'-'*50}\n")

agent.invoke("When Pokémon Gold and Silver was released?")
agent.invoke("Which one was the first 3D platformer Mario game?")
agent.invoke("Was Mortal Kombat X realeased for Playstation 5?")

Processing Query 1: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Query: When Pokémon Gold and Silver was released?
Agent Output: Pokémon Gold and Silver was released in 1999 for the Game Boy Color platform.
--------------------------------------------------

Processing Query 1: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Query: Which one was the first 3D platformer Mario game?
Agent Output: The first 3D platformer Mario game was "Super Mario 64," released in 1996 for the Nint

Run('30b3b098-a519-442b-81bd-992bae20cf8b')

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes